## UrbanRide_14 — Customer Segmentation Model
**Type:** Clustering — unsupervised learning, no label  
**Algorithm:** KMeans  
**Source:** `urbanride.gold.customer_features`  
**Schedule:** Weekly · Sunday 04:00 AM (Job 4)

**Key difference vs other models:**
- No label — KMeans finds natural groups in data
- StandardScaler is mandatory — KMeans uses distances, scale matters
- Elbow method to find optimal K
- Output = cluster_id per customer, then profiled and labelled

**Experiment:** `/urbanride_customer_segmentation`  
**Registry:** `urbanride.default.urbanride_segmentation_model`  
**Output:** `urbanride.gold.customer_segments`

In [0]:
# ── Run this cell first before anything else ─────────────────
import gc

stale_models = [
    'kmeans_model', 'final_model', 'BEST_MODEL',
    'kmeans_pipeline', 'final_pipeline'
]
for name in stale_models:
    try:
        del globals()[name]
        print(f'  Deleted: {name}')
    except KeyError:
        pass

gc.collect()
print('  Python memory cleared.')
print()

spark.sql('SELECT 1').show()
print('  Spark connection healthy.')

spark.sql('SHOW SCHEMAS IN urbanride').show()
print('  Catalog accessible.')

print()
print('Session ready. Safe to run notebook.')


  Python memory cleared.

+---+
|  1|
+---+
|  1|
+---+

  Spark connection healthy.
+------------------+
|      databaseName|
+------------------+
|            bronze|
|           default|
|              gold|
|information_schema|
|           landing|
|            silver|
+------------------+

  Catalog accessible.

Session ready. Safe to run notebook.


In [0]:
import mlflow
import mlflow.spark
from mlflow.tracking import MlflowClient
from mlflow.models.signature import infer_signature
from pyspark.ml import Pipeline
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.sql.functions import (
    col, avg, count, round as spark_round,
    min as spark_min, max as spark_max,
    lit, current_timestamp, when
)
import pandas as pd
import gc
import tempfile
import os

CATALOG         = 'urbanride'
GOLD            = f'{CATALOG}.gold'
EXPERIMENT_NAME = '/urbanride_customer_segmentation'
MODEL_NAME      = 'urbanride.default.urbanride_segmentation_model'

# TMP_DIR — MLflow staging area during model save
TMP_DIR = '/Volumes/urbanride/silver/mlflow_artifacts/mlflow_tmp'
try:
    dbutils.fs.ls(TMP_DIR)
    print(f'TMP_DIR already exists: {TMP_DIR}')
except Exception:
    dbutils.fs.mkdirs(TMP_DIR)
    print(f'TMP_DIR created: {TMP_DIR}')

# Features — behaviour based, all numeric
# StandardScaler mandatory — KMeans uses Euclidean distance
# total_spend (~35K) vs cancellation_rate (~0.08) — must be normalised
FEATURE_COLS = [
    'total_trips',
    'total_spend',
    'days_since_last_trip',
    'avg_trip_distance_km',
    'cancellation_rate',
    'avg_surge_multiplier',
    'weekend_trip_ratio',
    'failed_payment_count',
    'customer_age_days',
    'avg_spend_per_trip',
]

mlflow.set_experiment(EXPERIMENT_NAME)

print(f'Catalog    : {CATALOG}')
print(f'Source     : {GOLD}.customer_features')
print(f'Experiment : {EXPERIMENT_NAME}')
print(f'Model name : {MODEL_NAME}')
print(f'Features   : {len(FEATURE_COLS)}')
print('Config ready.')


2026/03/12 14:33:23 INFO mlflow.tracking.fluent: Experiment with name '/urbanride_customer_segmentation' does not exist. Creating a new experiment.


TMP_DIR already exists: /Volumes/urbanride/silver/mlflow_artifacts/mlflow_tmp
Catalog    : urbanride
Source     : urbanride.gold.customer_features
Experiment : /urbanride_customer_segmentation
Model name : urbanride.default.urbanride_segmentation_model
Features   : 10
Config ready.


In [0]:
print('Loading gold.customer_features...')

df_raw = spark.table(f'{GOLD}.customer_features')
total  = df_raw.count()
print(f'  Total customers : {total:,}')

# Select only feature columns + customer_id for traceability
# Drop rows with nulls in feature columns — KMeans cannot handle nulls
df = df_raw.select(['customer_id', 'city', 'is_churned'] + FEATURE_COLS) \
           .dropna(subset=FEATURE_COLS)

clean_count = df.count()
print(f'  After dropna    : {clean_count:,}')
print(f'  Dropped         : {total - clean_count:,}')
print()

print('Feature stats:')
df.select(FEATURE_COLS).describe().show()


Loading gold.customer_features...
  Total customers : 89,041
  After dropna    : 89,041
  Dropped         : 0

Feature stats:
+-------+------------------+------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------+------------------+
|summary|       total_trips|       total_spend|days_since_last_trip|avg_trip_distance_km|   cancellation_rate|avg_surge_multiplier|  weekend_trip_ratio|failed_payment_count|customer_age_days|avg_spend_per_trip|
+-------+------------------+------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------+------------------+
|  count|             89041|             89041|               89041|               89041|               89041|               89041|               89041|               89041|            89041|             89041|
|   mean|239.88662526251952|30

In [0]:
print('Building feature pipeline...')

# ── Why StandardScaler is mandatory for KMeans ────────────────
# KMeans groups customers by Euclidean distance
# total_spend = 35,000  vs  cancellation_rate = 0.08
# Without scaling — total_spend dominates distance calculation completely
# Every cluster would just be grouping by spend, ignoring all other features
# StandardScaler brings all features to mean=0, std=1 — equal contribution
# ─────────────────────────────────────────────────────────────

assembler = VectorAssembler(
    inputCols    = FEATURE_COLS,
    outputCol    = 'features_raw',
    handleInvalid= 'skip'
)

scaler = StandardScaler(
    inputCol = 'features_raw',
    outputCol= 'features',
    withMean = True,
    withStd  = True
)

# Fit assembler + scaler on full dataset — no train/test split for clustering
# Unsupervised — we use all 89K customers to find natural groups
prep_pipeline    = Pipeline(stages=[assembler, scaler])
prep_model       = prep_pipeline.fit(df)
df_features      = prep_model.transform(df)

print(f'  Features assembled and scaled.')
print(f'  Rows ready for KMeans : {df_features.count():,}')


Building feature pipeline...
  Features assembled and scaled.
  Rows ready for KMeans : 89,041


In [0]:
print('Running elbow method to find optimal K...')
print('Testing K = 2 to 8')
print()

# ── Elbow Method ──────────────────────────────────────────────
# WSSSE = Within Set Sum of Squared Errors
# Lower WSSSE = tighter clusters = better
# But WSSSE always decreases as K increases
# Optimal K = where the curve bends (diminishing returns)

# Silhouette score — how well separated clusters are
# Range: -1 to 1. Higher = better separated clusters
# ─────────────────────────────────────────────────────────────

silhouette_evaluator = ClusteringEvaluator(
    featuresCol    = 'features',
    predictionCol  = 'prediction',
    metricName     = 'silhouette'
)

elbow_results = []

for k in range(2, 9):
    kmeans = KMeans(
        featuresCol = 'features',
        k           = k,
        seed        = 42,
        maxIter     = 20,
    )

    with mlflow.start_run(run_name=f'kmeans_k{k}') as run:
        km_model    = kmeans.fit(df_features)
        km_preds    = km_model.transform(df_features)
        wssse       = km_model.summary.trainingCost
        silhouette  = silhouette_evaluator.evaluate(km_preds)

        mlflow.log_param('k',          k)
        mlflow.log_metric('wssse',     wssse)
        mlflow.log_metric('silhouette',silhouette)

        elbow_results.append({
            'k'         : k,
            'wssse'     : wssse,
            'silhouette': silhouette,
            'run_id'    : run.info.run_id
        })

        print(f'  K={k}  WSSSE={wssse:>15.2f}  Silhouette={silhouette:.4f}')

    # Delete model immediately — avoid cache overflow
    del km_model
    del km_preds
    gc.collect()

print()
print('Elbow results:')
print(f'  {"K":>4}  {"WSSSE":>15}  {"Silhouette":>12}')
print(f'  {"-"*35}')
for r in elbow_results:
    print(f'  {r["k"]:>4}  {r["wssse"]:>15.2f}  {r["silhouette"]:>12.4f}')

# Pick K with best silhouette score
best_elbow  = max(elbow_results, key=lambda x: x['silhouette'])
OPTIMAL_K   = best_elbow['k']
print()
print(f'Optimal K : {OPTIMAL_K}  (best silhouette = {best_elbow["silhouette"]:.4f}')


Running elbow method to find optimal K...
Testing K = 2 to 8

  K=2  WSSSE=      673443.91  Silhouette=0.7263
  K=3  WSSSE=      583535.75  Silhouette=0.2575
  K=4  WSSSE=      546892.22  Silhouette=0.2142
  K=5  WSSSE=      522090.56  Silhouette=0.1844
  K=6  WSSSE=      500328.72  Silhouette=0.1826
  K=7  WSSSE=      481431.61  Silhouette=0.1817
  K=8  WSSSE=      461337.60  Silhouette=0.1818

Elbow results:
     K            WSSSE    Silhouette
  -----------------------------------
     2        673443.91        0.7263
     3        583535.75        0.2575
     4        546892.22        0.2142
     5        522090.56        0.1844
     6        500328.72        0.1826
     7        481431.61        0.1817
     8        461337.60        0.1818

Optimal K : 2  (best silhouette = 0.7263


In [0]:
print(f'Training final KMeans with K={OPTIMAL_K}...')

kmeans_final = KMeans(
    featuresCol = 'features',
    k           = OPTIMAL_K,
    seed        = 42,
    maxIter     = 50,   # more iterations for final model
)

with mlflow.start_run(run_name=f'kmeans_final_k{OPTIMAL_K}') as run:
    BEST_MODEL  = kmeans_final.fit(df_features)
    df_clustered = BEST_MODEL.transform(df_features)

    final_wssse      = BEST_MODEL.summary.trainingCost
    final_silhouette = silhouette_evaluator.evaluate(df_clustered)

    mlflow.log_param('model_type',   'KMeans')
    mlflow.log_param('k',            OPTIMAL_K)
    mlflow.log_param('max_iter',     50)
    mlflow.log_param('feature_count',len(FEATURE_COLS))
    mlflow.log_metric('wssse',       final_wssse)
    mlflow.log_metric('silhouette',  final_silhouette)
    BEST_RUN_ID = run.info.run_id

print(f'  K              : {OPTIMAL_K}')
print(f'  WSSSE          : {final_wssse:.2f}')
print(f'  Silhouette     : {final_silhouette:.4f}')
print()

# Cluster size distribution
print('Cluster size distribution:')
df_clustered.groupBy('prediction').count() \
    .orderBy('prediction').show()


Training final KMeans with K=2...
  K              : 2
  WSSSE          : 673443.81
  Silhouette     : 0.7263

Cluster size distribution:
+----------+-----+
|prediction|count|
+----------+-----+
|         0| 6678|
|         1|82363|
+----------+-----+



In [0]:
print('Profiling clusters...')
print()

# Compute avg of each feature per cluster
# This tells us what makes each cluster different from others
profile = df_clustered.groupBy('prediction').agg(
    count('customer_id').alias('customer_count'),
    spark_round(avg('total_trips'),          1).alias('avg_trips'),
    spark_round(avg('total_spend'),          0).alias('avg_spend'),
    spark_round(avg('days_since_last_trip'), 1).alias('avg_days_inactive'),
    spark_round(avg('avg_spend_per_trip'),   0).alias('avg_spend_per_trip'),
    spark_round(avg('cancellation_rate'),    4).alias('avg_cancel_rate'),
    spark_round(avg('avg_surge_multiplier'), 4).alias('avg_surge'),
    spark_round(avg('weekend_trip_ratio'),   4).alias('avg_weekend_ratio'),
    spark_round(avg('failed_payment_count'), 2).alias('avg_failed_payments'),
    spark_round(avg('customer_age_days'),    0).alias('avg_tenure_days'),
    spark_round(
        avg(col('is_churned').cast('int')) * 100, 1
    ).alias('churn_rate_pct'),
).orderBy('prediction')

print('Cluster profiles:')
profile.show(truncate=False)

# Store profiles as pandas for labelling
profile_pd = profile.toPandas()
print('Profile ready. Use this to label segments in Cell 9.')


Profiling clusters...

Cluster profiles:
+----------+--------------+---------+---------+-----------------+------------------+---------------+---------+-----------------+-------------------+---------------+--------------+
|prediction|customer_count|avg_trips|avg_spend|avg_days_inactive|avg_spend_per_trip|avg_cancel_rate|avg_surge|avg_weekend_ratio|avg_failed_payments|avg_tenure_days|churn_rate_pct|
+----------+--------------+---------+---------+-----------------+------------------+---------------+---------+-----------------+-------------------+---------------+--------------+
|0         |6678          |126.8    |16296.0  |96.1             |145.0             |0.0864         |1.327    |0.2027           |3.42               |149.0          |100.0         |
|1         |82363         |249.1    |31771.0  |10.4             |144.0             |0.085          |1.3126   |0.2106           |6.83               |96.0           |5.0           |
+----------+--------------+---------+---------+------------

In [0]:
print('Profiling clusters...')
print()

# Compute avg of each feature per cluster
# This tells us what makes each cluster different from others
profile = df_clustered.groupBy('prediction').agg(
    count('customer_id').alias('customer_count'),
    spark_round(avg('total_trips'),          1).alias('avg_trips'),
    spark_round(avg('total_spend'),          0).alias('avg_spend'),
    spark_round(avg('days_since_last_trip'), 1).alias('avg_days_inactive'),
    spark_round(avg('avg_spend_per_trip'),   0).alias('avg_spend_per_trip'),
    spark_round(avg('cancellation_rate'),    4).alias('avg_cancel_rate'),
    spark_round(avg('avg_surge_multiplier'), 4).alias('avg_surge'),
    spark_round(avg('weekend_trip_ratio'),   4).alias('avg_weekend_ratio'),
    spark_round(avg('failed_payment_count'), 2).alias('avg_failed_payments'),
    spark_round(avg('customer_age_days'),    0).alias('avg_tenure_days'),
    spark_round(
        avg(col('is_churned').cast('int')) * 100, 1
    ).alias('churn_rate_pct'),
).orderBy('prediction')

print('Cluster profiles:')
# profile.show(truncate=False)
display(profile.limit(10))

# Store profiles as pandas for labelling
profile_pd = profile.toPandas()
print('Profile ready. Use this to label segments in Cell 9.')


Profiling clusters...

Cluster profiles:


prediction,customer_count,avg_trips,avg_spend,avg_days_inactive,avg_spend_per_trip,avg_cancel_rate,avg_surge,avg_weekend_ratio,avg_failed_payments,avg_tenure_days,churn_rate_pct
0,6678,126.8,16296.0,96.1,145.0,0.0864,1.327,0.2027,3.42,149.0,100.0
1,82363,249.1,31771.0,10.4,144.0,0.085,1.3126,0.2106,6.83,96.0,5.0


Profile ready. Use this to label segments in Cell 9.


In [0]:
print('Registering segmentation model to MLflow Model Registry...')

client = MlflowClient()

# Step 1 — Create model name in registry
try:
    client.create_registered_model(
        name        = MODEL_NAME,
        description = f'KMeans customer segmentation for Urbanride. K={OPTIMAL_K} segments.'
    )
    print(f'  Created model: {MODEL_NAME}')
except Exception:
    print(f'  Model already exists: {MODEL_NAME} — adding new version')

# Step 2 — Build signature
# Input  = 10 feature columns
# Output = prediction (cluster id 0 to K-1)
sample_input  = df_features.select(FEATURE_COLS).limit(100)
sample_pandas = sample_input.toPandas()

sample_preds  = BEST_MODEL.transform(
    prep_model.transform(sample_input)
)
sample_output = sample_preds.select('prediction').toPandas()

signature = infer_signature(
    model_input  = sample_pandas,
    model_output = sample_output
)
print('  Signature inferred.')
print(signature)

# Step 3 — Log and register
with mlflow.start_run(run_name='register_segmentation_model') as reg_run:
    mlflow.log_param('model_type',    'KMeans')
    mlflow.log_param('k',             OPTIMAL_K)
    mlflow.log_param('feature_count', len(FEATURE_COLS))
    mlflow.log_metric('wssse',        final_wssse)
    mlflow.log_metric('silhouette',   final_silhouette)

    # Save cluster profiles as artifact
    with tempfile.TemporaryDirectory() as tmp:
        profile_path = os.path.join(tmp, 'cluster_profiles.csv')
        profile_pd.to_csv(profile_path, index=False)
        mlflow.log_artifact(profile_path)
    print('  Cluster profiles logged as artifact.')

    mlflow.spark.log_model(
        spark_model   = BEST_MODEL,
        artifact_path = 'segmentation_model',
        signature     = signature,
        input_example = sample_pandas.head(5),
        dfs_tmpdir    = TMP_DIR
    )

    model_uri = f'runs:/{reg_run.info.run_id}/segmentation_model'
    mv = mlflow.register_model(model_uri, MODEL_NAME)

# Step 4 — Add version description
versions       = client.search_model_versions(f"name='{MODEL_NAME}'")
latest_version = max([int(v.version) for v in versions])

client.update_model_version(
    name        = MODEL_NAME,
    version     = str(latest_version),
    description = (
        f'KMeans. K={OPTIMAL_K}. '
        f'Silhouette={final_silhouette:.4f}. '
        f'Trained on 89K ZipCab customers. '
        f'Features: trips, spend, recency, distance, cancellation rate.'
    )
)
print(f'  Description added to {MODEL_NAME} v{latest_version}')

print()
print(f'  Model registered : {MODEL_NAME}')
print(f'  Version          : {latest_version}')
print(f'  Load URI         : models:/{MODEL_NAME}/{latest_version}')


Registering segmentation model to MLflow Model Registry...
  Created model: urbanride.default.urbanride_segmentation_model


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


  Signature inferred.
inputs: 
  ['total_trips': long (required), 'total_spend': double (required), 'days_since_last_trip': integer (required), 'avg_trip_distance_km': double (required), 'cancellation_rate': double (required), 'avg_surge_multiplier': double (required), 'weekend_trip_ratio': double (required), 'failed_payment_count': long (required), 'customer_age_days': integer (required), 'avg_spend_per_trip': double (required)]
outputs: 
  ['prediction': integer (required)]
params: 
  None

  Cluster profiles logged as artifact.


2026/03/12 14:39:00 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/12 14:39:04 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-860ea08f-05ba-41fe-8142-b1/tmpwecqo3ge/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/03/12 14:39:04 WARNING mlflow.models.model: Failed to validate serving input example {
  "dataframe_split": {
    "columns": [
      "total_trips",
      "total_spend",
      "days_since_last_trip",
      "avg_trip_distance_km",
      "cancellation_rate"

  Description added to urbanride.default.urbanride_segmentation_model v1

  Model registered : urbanride.default.urbanride_segmentation_model
  Version          : 1
  Load URI         : models:/urbanride.default.urbanride_segmentation_model/1


In [0]:
print('Saving customer segments to Delta table...')

# Map cluster id to human readable segment label
# Labels derived from Cell 6 cluster profiles
# Adjust these labels after reviewing Cell 6 output
# Default mapping — update based on actual profile values
# segment_labels = {
#     0: 'Segment_0',
#     1: 'Segment_1',
#     2: 'Segment_2',
#     3: 'Segment_3',
#     4: 'Segment_4',
# }
segment_labels = {
    0: 'Churned',
    1: 'Active',
}

# Build segment label column from cluster id
segment_expr = when(col('prediction') == 0, segment_labels[0])
for k in range(1, OPTIMAL_K):
    label = segment_labels.get(k, f'Segment_{k}')
    segment_expr = segment_expr.when(col('prediction') == k, label)
segment_expr = segment_expr.otherwise('Unknown')

df_segments = df_clustered.select(
    col('customer_id'),
    col('city'),
    col('is_churned'),
    col('prediction').alias('cluster_id'),
    segment_expr.alias('segment_label'),
    col('total_trips'),
    col('total_spend'),
    col('days_since_last_trip'),
    col('avg_spend_per_trip'),
    col('cancellation_rate'),
    col('customer_age_days'),
).withColumn('_processed_at',  current_timestamp()) \
 .withColumn('_model_version', lit(str(latest_version))) \
 .withColumn('_model_name',    lit(MODEL_NAME))

df_segments.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{GOLD}.customer_segments')

seg_count = df_segments.count()
print(f'  Segments written : {seg_count:,}')
print(f'  Table            : {GOLD}.customer_segments')


Saving customer segments to Delta table...
  Segments written : 89,041
  Table            : urbanride.gold.customer_segments


In [0]:
print('=== SEGMENTATION MODEL VERIFICATION ===')
print()

df_verify = spark.table(f'{GOLD}.customer_segments')
total     = df_verify.count()

print(f'  Total customers : {total:,}')
print(f'  K               : {OPTIMAL_K}')
print(f'  Silhouette      : {final_silhouette:.4f}')
print()

# Segment size distribution
print('Segment distribution:')
df_verify.groupBy('cluster_id', 'segment_label') \
    .count() \
    .orderBy('cluster_id').show()

# Segment profiles — key metrics per segment
print('Segment profiles (key metrics):')
df_verify.groupBy('cluster_id', 'segment_label').agg(
    count('customer_id').alias('customers'),
    spark_round(avg('total_trips'),          1).alias('avg_trips'),
    spark_round(avg('total_spend'),          0).alias('avg_spend'),
    spark_round(avg('days_since_last_trip'), 1).alias('avg_days_inactive'),
    spark_round(avg('cancellation_rate'),    4).alias('avg_cancel_rate'),
).orderBy('cluster_id').show(truncate=False)

# Churn rate per segment — which segment is most at risk?
print('Churn rate by segment:')
df_verify.groupBy('cluster_id', 'segment_label').agg(
    spark_round(
        avg(col('is_churned').cast('int')) * 100, 1
    ).alias('churn_rate_pct'),
    count('customer_id').alias('customers')
).orderBy('churn_rate_pct', ascending=False).show(truncate=False)

# City distribution per segment
print('City distribution per segment:')
df_verify.groupBy('segment_label', 'city') \
    .count() \
    .orderBy('segment_label', 'count', ascending=[True, False]).show(20, truncate=False)

print()
print('='*55)
print('Segmentation model complete.')
print(f'Experiments UI : {EXPERIMENT_NAME}')
print(f'Models UI      : {MODEL_NAME}')
print(f'Segments table : {GOLD}.customer_segments')
print()
print('ALL 4 ML MODELS COMPLETE:')
print('  11 — Churn Prediction     → urbanride.gold.churn_predictions')
print('  12 — Fraud Detection      → urbanride.gold.fraud_predictions')
print('  13 — Demand Forecasting   → urbanride.gold.demand_predictions')
print('  14 — Customer Segmentation→ urbanride.gold.customer_segments')
print()
print('Next : SQL Dashboards')
print('='*55)


=== SEGMENTATION MODEL VERIFICATION ===

  Total customers : 89,041
  K               : 2
  Silhouette      : 0.7263

Segment distribution:
+----------+-------------+-----+
|cluster_id|segment_label|count|
+----------+-------------+-----+
|         0|      Churned| 6678|
|         1|       Active|82363|
+----------+-------------+-----+

Segment profiles (key metrics):
+----------+-------------+---------+---------+---------+-----------------+---------------+
|cluster_id|segment_label|customers|avg_trips|avg_spend|avg_days_inactive|avg_cancel_rate|
+----------+-------------+---------+---------+---------+-----------------+---------------+
|0         |Churned      |6678     |126.8    |16296.0  |96.1             |0.0864         |
|1         |Active       |82363    |249.1    |31771.0  |10.4             |0.085          |
+----------+-------------+---------+---------+---------+-----------------+---------------+

Churn rate by segment:
+----------+-------------+--------------+---------+
|cluste